In [ ]:
import hardware
import matplotlib.pyplot as plt
import numpy as np
import torch
from env import SKILLS, DuckEnv
from rollout import latest_checkpoint, load_checkpoint

hw = hardware.load("2026-11-07-hd1910")
policy, ckpt = load_checkpoint(latest_checkpoint("m04-walk"), "cuda")   # 上一讲训的走路策略
env = DuckEnv(SKILLS["walk"], hw, num_envs=1, device="cuda")
env.set_curriculum_step(ckpt["common_step_counter"])                     # 课程表对齐到训练到的那一档
rewards = env.unwrapped.reward_manager
print(rewards)                                                           # 走路任务的奖励项与权重

In [ ]:
def walk(steps=300):
    env.reset()
    env.pin_commands(0.3)                                # 一直前进 0.3 m/s
    obs, _ = env.get_observations()
    log = []
    with torch.no_grad():
        for _ in range(steps):                           # 300 步 = 6 秒
            obs, _, reward, _, _ = env.step(policy.act_inference(obs))
            terms = {name: v[0] for name, v in rewards.get_active_iterable_terms(0)}   # 各项：权重 × 原始得分（每秒）
            log.append((reward.item(), terms))
    return log


log = walk()
reward_6s, terms_6s = log[-1]                            # 第 6 秒这一步

In [ ]:
dt = env.unwrapped.step_dt                               # 控制周期 0.02 秒
dt * sum(terms_6s.values()), reward_6s                   # 各项相加、乘 0.02，就是环境给的这一步奖励

In [ ]:
sorted(terms_6s.items(), key=lambda kv: -abs(kv[1]))[:6] # 这一步挣分最多的几项

In [ ]:
w = rewards.get_term_cfg("track_linear_velocity").weight
terms_6s["track_linear_velocity"] / w                    # 线速度跟踪的原始得分：在 0 到 1 之间

In [ ]:
e = np.linspace(-1, 1, 201)                              # 速度误差（m/s）
plt.plot(e, np.exp(-e**2 / 0.1), label="exp(-e^2 / sigma^2)")   # 钟形：误差越小越接近 1
plt.plot(e, -e**2 / 0.1, label="-e^2 / sigma^2")         # 负平方：误差越大扣得越多
plt.ylim(-3, 1.2)
plt.legend()

In [ ]:
before = sum(r for r, _ in log)                          # 原来的权重下，6 秒的奖励之和
rewards.get_term_cfg("air_time").weight = 0.0            # 把滞空这一项的权重改成 0
after = sum(r for r, _ in walk())                        # 同一个策略再走 6 秒
before, after                                            # 少了滞空那一份